# 03 — Size Estimation Validation

Validates the knuckle-based mango size formula:
```
real_size_cm = (mango_px / knuckle_span_px) x HAND_SPAN_CM
```
Compares estimated sizes against manually measured ground truth.

**Target:** MAE < 0.5 cm for reliable growth stage assignment.

## 1. Setup

In [ ]:
!pip install mediapipe ultralytics opencv-python matplotlib numpy

import cv2, os
import numpy as np
import mediapipe as mp
import matplotlib.pyplot as plt
from ultralytics import YOLO

YOLO_WEIGHTS = '../weights/mango_yolov8.pt'
TEST_IMG_DIR = '../datasets/raw'
HAND_SPAN_CM = 5.4  # index-to-pinky MCP span in cm (3 knuckle widths)

## 2. Ground truth measurements

Fill in manually measured mango widths (cm) for each test image.
Measure the widest diameter with a ruler beside the mango.

In [ ]:
# Format: 'filename.jpg': actual_size_cm
GROUND_TRUTH = {
    # 'test_001.jpg': 3.5,
    # 'test_002.jpg': 2.1,
}

## 3. Load models

In [ ]:
yolo = YOLO(YOLO_WEIGHTS)
hands = mp.solutions.hands.Hands(static_image_mode=True, max_num_hands=1, min_detection_confidence=0.7)
print('Models loaded.')

## 4. Detection helpers

In [ ]:
def get_knuckle_px(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    res = hands.process(rgb)
    if not res.multi_hand_landmarks:
        return None
    lm = res.multi_hand_landmarks[0].landmark
    h, w = img_bgr.shape[:2]
    ix = lm[mp.solutions.hands.HandLandmark.INDEX_FINGER_MCP].x * w
    px = lm[mp.solutions.hands.HandLandmark.PINKY_MCP].x * w
    return abs(ix - px)

def get_mango_size(img_bgr, knuckle_px):
    res = yolo.predict(img_bgr, verbose=False)[0]
    if not res.boxes:
        return None
    x1,y1,x2,y2 = res.boxes[0].xyxy[0].tolist()
    mango_px = min(x2-x1, y2-y1)
    return round((mango_px / knuckle_px) * HAND_SPAN_CM, 2)

print('Helpers defined.')

## 5. Run validation

In [ ]:
log = []
for fname, actual in GROUND_TRUTH.items():
    path = os.path.join(TEST_IMG_DIR, fname)
    if not os.path.exists(path):
        print(f'SKIP {fname} — not found')
        continue
    img = cv2.imread(path)
    kpx = get_knuckle_px(img)
    if kpx is None:
        print(f'SKIP {fname} — no hand detected')
        continue
    est = get_mango_size(img, kpx)
    if est is None:
        print(f'SKIP {fname} — no mango detected')
        continue
    err = abs(est - actual)
    log.append({'file':fname,'actual':actual,'estimated':est,'error':err})
    print(f'{fname}: actual={actual}cm  estimated={est}cm  error={err:.2f}cm')
print(f'\nProcessed {len(log)}/{len(GROUND_TRUTH)} images')

## 6. Error metrics

In [ ]:
if log:
    errors = [r['error'] for r in log]
    mae  = np.mean(errors)
    rmse = np.sqrt(np.mean([e**2 for e in errors]))
    print(f'MAE  : {mae:.3f} cm')
    print(f'RMSE : {rmse:.3f} cm')
    print(f'Max  : {max(errors):.3f} cm')
    print()
    if mae < 0.5:
        print('MAE < 0.5 cm — size estimation is accurate enough.')
    else:
        print('MAE >= 0.5 cm — tune HAND_SPAN_CM or improve capture instructions.')
else:
    print('No results. Fill GROUND_TRUTH and re-run.')

## 7. Find optimal HAND_SPAN_CM

In [ ]:
if log:
    best_span, best_mae = None, float('inf')
    for span in np.arange(3.0, 8.0, 0.1):
        errs = []
        for r in log:
            img = cv2.imread(os.path.join(TEST_IMG_DIR, r['file']))
            kpx = get_knuckle_px(img)
            res = yolo.predict(img, verbose=False)[0]
            if not res.boxes or kpx is None: continue
            x1,y1,x2,y2 = res.boxes[0].xyxy[0].tolist()
            est = (min(x2-x1,y2-y1) / kpx) * span
            errs.append(abs(est - r['actual']))
        if errs and np.mean(errs) < best_mae:
            best_mae = np.mean(errs)
            best_span = round(span, 1)
    print(f'Optimal HAND_SPAN_CM : {best_span} cm  (MAE={best_mae:.3f} cm)')
    print(f'Current              : {HAND_SPAN_CM} cm')
    if best_span and abs(best_span - HAND_SPAN_CM) > 0.3:
        print(f'Update HAND_SPAN_CM to {best_span} in mlharum-api/services/size_estimator.py')

## 8. Visualise predictions

In [ ]:
samples = log[:4]
if samples:
    fig, axes = plt.subplots(1, len(samples), figsize=(20, 5))
    for ax, r in zip(axes if len(samples)>1 else [axes], samples):
        img = cv2.imread(os.path.join(TEST_IMG_DIR, r['file']))
        res = yolo.predict(img, verbose=False)[0]
        ax.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB))
        ax.set_title(f"actual={r['actual']}cm  est={r['estimated']}cm", fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Fill GROUND_TRUTH to visualise.')